In [6]:
import numpy as np
from vd_selectors import VD_LARS, VD_OMP, VD_AFS, VDOptions, VDDummyLaw

def standardize_X(X):
    X = np.asarray(X, dtype=float).copy(order="F")
    X -= X.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(X, axis=0)
    norms[norms == 0] = 1.0
    X /= norms
    return np.asfortranarray(X), norms

def center_y(y):
    y = np.asarray(y, dtype=float).copy()
    y -= y.mean()
    return y

np.random.seed(42)

# moderately high-dimensional signal
n, p = 300, 1000
X_raw = np.random.randn(n, p)
beta_true = np.zeros(p)
beta_true[:3] = [2.0, -1.25, 0.8]
y_raw = X_raw @ beta_true + 0.5 * np.random.randn(n)

X, norms = standardize_X(X_raw)
y = center_y(y_raw)

opt = VDOptions()
opt.T_max = 50
opt.max_vd_proj = 100
opt.eps = 1e-10
opt.seed = 124
opt.dummy_law = VDDummyLaw.Spherical

# If rho is part of VDOptions, uncomment and set it here for AFS
opt.rho = 0.5

L = 5 * p

lars = VD_LARS(X, y, L, opt)
omp  = VD_OMP(X, y, L, opt)
afs  = VD_AFS(X, y, L, opt)

print("Constructed OK")
print("LARS:", lars.n_samples(), lars.n_features(), lars.num_dummies())
print("OMP :", omp.n_samples(),  omp.n_features(),  omp.num_dummies())
print("AFS :", afs.n_samples(),  afs.n_features(),  afs.num_dummies())

path_lars = lars.run(1)
path_omp  = omp.run(1)
path_afs  = afs.run(1)

print("LARS path shape:", path_lars.shape)
print("OMP  path shape:", path_omp.shape)
print("AFS  path shape:", path_afs.shape)

print("LARS realized dummies:", lars.num_realized_dummies())
print("OMP  realized dummies:", omp.num_realized_dummies())
print("AFS  realized dummies:", afs.num_realized_dummies())

print("LARS active features:", lars.active_features())
print("OMP  active features:", omp.active_features())
print("AFS  active features:", afs.active_features())

print("LARS beta[:10]:", lars.beta_real()[:10])
print("OMP  beta[:10]:", omp.beta_real()[:10])
print("AFS  beta[:10]:", afs.beta_real()[:10])

# simple checks
assert path_lars.shape[0] == p
assert path_omp.shape[0] == p
assert path_afs.shape[0] == p

assert lars.num_realized_dummies() >= 1
assert omp.num_realized_dummies() >= 1
assert afs.num_realized_dummies() >= 1

# first 3 reals should usually appear early on this synthetic problem
print("Top LARS support:", np.flatnonzero(np.abs(lars.beta_real()) > 1e-10)[:10])
print("Top OMP  support:", np.flatnonzero(np.abs(omp.beta_real()) > 1e-10)[:10])
print("Top AFS  support:", np.flatnonzero(np.abs(afs.beta_real()) > 1e-10)[:10])

Constructed OK
LARS: 300 1000 5000
OMP : 300 1000 5000
AFS : 300 1000 5000
LARS path shape: (1000, 4)
OMP  path shape: (1000, 5)
AFS  path shape: (1000, 8)
LARS realized dummies: 1
OMP  realized dummies: 1
AFS  realized dummies: 1
LARS active features: [('real', 0), ('real', 1), ('real', 2), ('dummy', 0)]
OMP  active features: [('real', 0), ('real', 1), ('real', 2), ('dummy', 0)]
AFS  active features: [('real', 0), ('real', 1), ('real', 2), ('dummy', 0)]
LARS beta[:10]: [ 34.83145482 -19.59321681  12.54303901   0.           0.
   0.           0.           0.           0.           0.        ]
OMP  beta[:10]: [ 36.37862828 -21.26347703  14.44144895   0.           0.
   0.           0.           0.           0.           0.        ]
AFS  beta[:10]: [ 36.12938543 -20.86431039  13.68543213   0.           0.
   0.           0.           0.           0.           0.        ]
Top LARS support: [0 1 2]
Top OMP  support: [0 1 2]
Top AFS  support: [0 1 2]
